# ViT-B/16 orig_in21k_ft_in1k — DIMER E2E supervised adaptation tutorial: a new head beyond ImageNet-1k, linear probe vs bounded unfreeze (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/vit-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/vit-classification-pipeline/blob/main/tutorials/vit_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Fvit__base__patch16__224.orig__in21k__ft__in1k-ffcc4d?style=flat)](https://huggingface.co/timm/vit_base_patch16_224.orig_in21k_ft_in1k) [![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Fpytorch--image--models-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/pytorch-image-models) [![arXiv](https://img.shields.io/badge/arXiv-2010.11929-b31b1b.svg)](https://arxiv.org/abs/2010.11929)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** ImageNet-1k single-label image classification (argmax plus top-k softmax scores) and bounded supervised adaptation to a label space the checkpoint does not have — a new linear head on the frozen 768-d pre-logits with an optional unfreeze of the last transformer blocks — measured by held-out accuracy and macro-F1, using the pinned ViT-B/16 `orig_in21k_ft_in1k` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/vit_classification_pipeline/`, at revision `7eaf4b1012d9`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `e0bd370de6799e8d1f47a911174ff4c3708e2323` (~346 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned ViT-B/16 `orig_in21k_ft_in1k` snapshot (safetensors, 346 MB), fetches 180 digest-pinned CC0 iNaturalist photographs of six bird species from the iNaturalist open-data bucket (19 MB, no credential), validates them and draws 108 / 24 / 48 training, validation and test images by a seeded stratified split, classifies three test images through the ImageNet-1k inference contract with an input manifest and a rejection probe, scores the frozen 768-d pre-logits on the test split by a majority floor, a cosine 5-NN vote and a linear probe (the **frozen policy**), trains a bounded unfreeze of the last two transformer blocks with the head and selects between it and the probe by validation log-loss (the **unfrozen policy**), scores the held-out split with the selected model, prints predictions before and after, exports the head and any trained blocks as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about two minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled images as a `.zip` holding `labels.csv` (columns `id`, `file`, `label`) beside the image files — images are decoded from the archive, never extracted to disk. They pass through the same validation, seeded stratified image-disjoint split, floors, frozen-policy probe, unfrozen-policy training and selection, held-out evaluation, prediction, artifact export and reload-parity cells as the iNaturalist sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the Vision Transformer splits one normalised 3×224×224 tensor into 196 patches of 16×16 pixels, prepends a class token, runs 12 self-attention blocks and reads 1000 logits from the class token in a single forward pass; `predict` applies a softmax and reports the argmax class plus the top-k classes with their scores. The input size is fixed at 224×224 by the checkpoint, so every image is resized and centre-cropped to it. The carried pipeline module adds snapshot verification, input validation, a fixed output contract and the `top_k_accuracy`, `validate_inputs` and `evaluation_report` helpers. **The label space is fixed to the 1000 ImageNet-1k classes:** an image whose subject is outside it still receives a label, with a score that says nothing about the mismatch.

What this notebook adds to inference is **supervised adaptation to a label space the checkpoint does not have, under an explicit frozen-vs-unfrozen policy**. The dataset is real: 180 CC0-licensed, research-grade iNaturalist photographs of six common North American birds (30 per species, one per observer per species), chosen a priori and pinned by photo id, byte size and SHA-256, fetched from the iNaturalist open-data bucket at run time and refused on any mismatch; every record keeps its observation URL and observer login. ImageNet-1k names three of the six (goldfinch, house finch, junco) and has no class for the three sparrows — Section 5 shows what the ImageNet head does with them. The carried `metrics.py` scores predictions by **accuracy** and **macro-F1** with per-class recall and a confusion matrix; a **majority floor** and a **cosine 5-NN vote** over the frozen 768-d pre-logits (the class token after the final norm, the vector the ImageNet head reads) frame the numbers. The **frozen policy** trains only a new six-way linear head on those frozen pre-logits (a linear probe); the **unfrozen policy** continues from that probe by training the last transformer blocks with the new head end to end, and the epoch with the lowest validation log-loss — which may be the probe itself — is kept. The ImageNet head is never trained or exported; `predict` keeps it. The adaptation question is whether unfreezing buys anything over the probe on 108 training photographs. Nothing here is a quality claim about your images: it is one seeded split of one small corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real image corpus and validate and split it without leakage; classify images through the public ImageNet-1k API and read the argmax decision and the uncalibrated top-k scores correctly; read accuracy and macro-F1 beside a majority floor and a k-NN baseline; train a linear probe on the frozen pre-logits and a bounded unfreeze with explicit hyperparameters and validation-based selection between the two policies; evaluate on an independent test split; compare predictions before and after; and export a safetensors adapter (head plus any trained blocks) that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** object detection, segmentation, multi-label tagging, OCR, open-vocabulary or zero-shot classification, attention maps, data augmentation, full-backbone or patch-embedding training, any training of the ImageNet-1k head, any ImageNet-1k accuracy claim, and any claim that six bird species from one photo site stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; float32 on both. Each 224 px image costs about 16.9 GMACs (upstream README): the build record measured about 0.04 s per image on CPU (6.4 s for the 180-image k-NN pass), a 7 s linear probe including feature extraction, and about 6 s per unfreeze epoch over 108 images plus a 24-image validation pass. The pinned `torch==2.14.0` install and the 346 MB checkpoint are the large downloads of the run, then the 19 MB of photographs.
- **Knowledge:** basic Python and NumPy; what a softmax over class logits is; what a linear probe is and why it is the cheapest honest test of a representation; what accuracy and macro-F1 measure and why macro-F1 punishes a forgotten class; what validation-based selection between two policies means.
- **Data contract:** records are `{{id, image, label}}` — a PIL image (or a path to one) with each side 1..4,096 px (the pipeline's own ceiling; images are resized and centre-cropped to 224 px, never rejected for being small), a label of 1..64 plain characters, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records and 2..100 classes; images are de-duplicated by decoded-pixel digest before splitting so the same photograph never sits in two splits. BYOD accepts a `.zip` (or a directory) holding `labels.csv` and the image files.
- **Validation is structural, not semantic:** nothing checks that a label is right for its image — a mislabelled set is trained on without complaint; observers can contribute to more than one split (the notebook counts them) because the sample is stratified by species, not by observer.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of people or private places are exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches 180 pinned objects (`<photo id>/medium.jpg` or `medium.jpeg`, each photo's own served extension recorded in the table, 19,183,071 bytes in total, one SHA-256 each in the carried `SAMPLE_RECORDS` table) from `inaturalist-open-data.s3.amazonaws.com` over HTTPS, each refused on any byte-size or SHA-256 mismatch before it is decoded; every photograph is CC0 by its own iNaturalist licence code and credited to its observer in the records.
- **External access:** the Hugging Face Hub only, to fetch the pinned `timm/vit_base_patch16_224.orig_in21k_ft_in1k` snapshot (~346 MB in total) at revision `e0bd370de679…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchaudio==2.11.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'vit-classification-pipeline',
    'repository_revision': '7eaf4b1012d9fa69c82f99508099b0089da79544',
    'embedded_module': 'src/vit_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/vit_classification_pipeline/metrics.py', 'src/vit_classification_pipeline/pipeline.py', 'src/vit_classification_pipeline/samples.py'],
    'module_sha256': '02abe4ed29a089d60f0f7d9c791a3311827fd8f220cbbb0f31cfb6b0f90b80b3',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/vit_classification_pipeline/` @ `7eaf4b1012d9`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/vit_classification_pipeline/metrics.py`

In [ ]:
"""Classification metrics over a labelled-image dataset, the majority baseline and a k-NN baseline.

Predictions are compared with gold labels: **accuracy**, **macro-F1** (the unweighted mean of per-class F1, so
a rare class counts as much as a common one), per-class precision / recall / F1 / support and the confusion
matrix. The **majority baseline** predicts the most frequent training label for every image; the **k-NN
baseline** labels each test image by a cosine vote of its `k` nearest training images in the frozen feature
space — what the checkpoint's representation gives with no training at all.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

import numpy as np

METRIC_DEFINITIONS = {
    "accuracy": "fraction of images whose predicted label equals the gold label",
    "macro_f1": "unweighted mean over classes of the per-class F1 (precision-recall harmonic mean)",
    "per_class": "precision, recall, F1 and support for every class of the gold label set",
    "confusion": "rows are gold classes, columns predicted classes, in the order of `classes`",
    "log_loss": "mean negative log-probability the head assigns to the gold label (the selection signal)",
}


def classification_metrics(
    y_true: Sequence[str], y_pred: Sequence[str], classes: Sequence[str]
) -> dict[str, Any]:
    """Accuracy, macro-F1, per-class scores and the confusion matrix over a fixed class order."""
    if not y_true:
        raise ValueError("no images to score")
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must align")
    order = list(classes)
    index = {c: i for i, c in enumerate(order)}
    unknown = [y for y in list(y_true) + list(y_pred) if y not in index]
    if unknown:
        raise ValueError(f"label {unknown[0]!r} is not in classes")
    confusion = np.zeros((len(order), len(order)), dtype=np.int64)
    for t, p in zip(y_true, y_pred, strict=True):
        confusion[index[t], index[p]] += 1
    per_class = {}
    f1s = []
    for i, name in enumerate(order):
        tp = int(confusion[i, i])
        fp = int(confusion[:, i].sum() - tp)
        fn = int(confusion[i, :].sum() - tp)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[name] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": int(confusion[i, :].sum()),
        }
        f1s.append(f1)
    return {
        "n": len(y_true),
        "accuracy": float(np.trace(confusion) / len(y_true)),
        "macro_f1": float(sum(f1s) / len(f1s)),
        "per_class": per_class,
        "confusion": confusion.tolist(),
        "classes": order,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def majority_baseline(
    train_labels: Sequence[str], test_labels: Sequence[str], classes: Sequence[str]
) -> dict[str, Any]:
    """Predict the most frequent training label (ties by class order) for every test image."""
    if not train_labels:
        raise ValueError("no training labels")
    counts = {c: 0 for c in classes}
    for label in train_labels:
        counts[label] = counts.get(label, 0) + 1
    majority = max(classes, key=lambda c: (counts.get(c, 0), -list(classes).index(c)))
    result = classification_metrics(test_labels, [majority] * len(test_labels), classes)
    result["baseline"] = f"majority training label ({majority!r}) predicted for every image"
    return result


def knn_predict(
    train_features: np.ndarray, train_labels: Sequence[str], test_features: np.ndarray, *, k: int = 5
) -> list[str]:
    """Cosine k-NN vote (unit-normalised rows assumed; normalised here anyway); ties by summed similarity."""
    if k < 1:
        raise ValueError("k must be positive")
    x = np.asarray(train_features, dtype=np.float32)
    q = np.asarray(test_features, dtype=np.float32)
    if x.ndim != 2 or q.ndim != 2 or x.shape[1] != q.shape[1]:
        raise ValueError("features must be 2-D with the same width")
    if len(train_labels) != x.shape[0]:
        raise ValueError("train_labels must align with train_features")
    x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    q = q / np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
    sims = q @ x.T
    k = min(k, x.shape[0])
    out = []
    labels = list(train_labels)
    for row in sims:
        top = np.argsort(-row, kind="stable")[:k]
        votes: dict[str, float] = {}
        for j in top:
            votes[labels[j]] = votes.get(labels[j], 0.0) + float(row[j])
        out.append(max(votes, key=lambda c: (votes[c], -labels.index(c))))
    return out


def knn_baseline(
    train_features: np.ndarray,
    train_labels: Sequence[str],
    test_features: np.ndarray,
    test_labels: Sequence[str],
    classes: Sequence[str],
    *,
    k: int = 5,
) -> dict[str, Any]:
    """k-NN over frozen features — what the representation gives without any training."""
    result = classification_metrics(
        test_labels, knn_predict(train_features, train_labels, test_features, k=k), classes
    )
    result["baseline"] = f"cosine {k}-NN vote over the frozen features of the training images"
    result["k"] = k
    return result

**Module 2/3:** `src/vit_classification_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""ImageNet-1k classification with the pinned ``timm/vit_base_patch16_224.orig_in21k_ft_in1k`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or,
when explicitly allowed, from the Hugging Face Hub at the pinned revision. Preprocessing is
the upstream ``pretrained_cfg`` (resize/crop/normalize) resolved through ``timm.data``: the
ViT-B/16 input is fixed at 3x224x224, so every image is resized and center-cropped to that size.

The adaptation contract (``features``, ``knn_baseline``, ``adapt``, ``evaluate``, ``classify``,
``save_artifact``, ``from_artifact``) trains a new linear head on the frozen 768-d pre-logits of a validated
``{id, image, label}`` dataset (the **frozen policy**), optionally continues with a bounded unfreeze of the
last transformer blocks (the **unfrozen policy**), scores held-out images by accuracy and macro-F1, and
exports the new head plus any trained blocks as a safetensors adapter bound to the pinned base weights. The
ImageNet-1k head and ``predict`` are unchanged by it, but ``predict`` reads the adapted backbone once an
unfreeze has run.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "timm/vit_base_patch16_224.orig_in21k_ft_in1k"
MODEL_REVISION = "e0bd370de6799e8d1f47a911174ff4c3708e2323"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "vit-base-p16-224"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

NUM_CLASSES = 1000
INPUT_SIZE = 224  # pixels; fixed_input_size per the checkpoint config; no dynamic-resolution path
MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MAX_BATCH = 64  # images per predict() call
DEFAULT_TOP_K = 5
DECISION_RULE = "argmax"  # the label reported as `predicted_index` is the softmax argmax; no threshold
WEIGHT_SHA256 = (
    "669b949ea91fd19217f200cee259780bde32210c1eb9a5af3859f0dd8346b2ec"  # manifest digest of WEIGHTS_FILE
)
PARAMETER_COUNT = 86_567_656
FEATURE_DIM = 768  # pre-logits width (the class token after the final norm)
TRANSFORMER_BLOCKS = 12  # ViT-B/16 depth
DEFAULT_TRAINABLE_BLOCKS = (
    2  # the unfrozen policy trains the last two blocks (14,175,744 parameters) after the probe
)
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.vit-base-p16-224.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
POLICY_FROZEN = "frozen backbone + linear probe"
POLICY_UNFROZEN = "unfrozen last {k} blocks + linear head"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _hub_reference(model_id: str, revision: str) -> str:
    """timm's ``hf-hub:owner/name@revision`` form; ``hf_split`` passes ``revision=`` to hf_hub_download."""
    return f"hf-hub:{model_id}@{revision}"


def top_k_accuracy(predictions: Sequence[Any], targets: Sequence[int], k: int = 1) -> float:
    """Fraction of items whose target index is among the first ``k`` predicted indices.

    ``predictions`` may be the per-image dicts returned by ``predict`` or plain index sequences.
    """
    if len(predictions) != len(targets):
        raise ValueError("predictions and targets must have the same length")
    if not predictions:
        raise ValueError("predictions must not be empty")
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer")
    hits = 0
    for pred, target in zip(predictions, targets, strict=True):
        ranked = pred["top_k"] if isinstance(pred, Mapping) else pred
        indices = [int(item["index"]) if isinstance(item, Mapping) else int(item) for item in ranked]
        hits += int(target in indices[:k])
    return hits / len(predictions)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "batch": [1, MAX_BATCH],
    "top_k": [1, NUM_CLASSES],
    "preprocessing": (
        "resize shorter side to 248 px, center-crop 224x224 (crop_pct 0.9, bicubic; fixed input size), "
        "normalise with mean 0.5 / std 0.5 per channel"
    ),
}


def _check_inputs(images: Any, top_k: int) -> list[Image.Image]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the images as a list."""
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, Sequence) or isinstance(images, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if not 1 <= len(images) <= MAX_BATCH:
        raise ValueError(f"batch size must be between 1 and MAX_BATCH={MAX_BATCH}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        width, height = image.size
        if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
            raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= NUM_CLASSES:
        raise ValueError(f"top_k must be between 1 and NUM_CLASSES={NUM_CLASSES}")
    return list(images)


def validate_inputs(
    images: Image.Image | Sequence[Image.Image],
    top_k: int = DEFAULT_TOP_K,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``predict`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked = _check_inputs(images, top_k)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)}
            for i, image in enumerate(checked)
        ],
        "top_k": top_k,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], targets: Sequence[int] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``targets`` (one ImageNet-1k index per prediction) the report carries ``top_k_accuracy``
    at k=1 and k=5 as sample-sanity evidence; without them the verdict is ``not-measurable`` and
    the report says what labelled data would make the task measurable.
    """
    predictions = result["predictions"]
    base = {
        "task": "imagenet-1k single-label classification",
        "decision_rule": result.get("decision_rule", DECISION_RULE),
        "sample_kind": sample_kind,
        "n_predictions": len(predictions),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if targets is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth class index was supplied for the evaluated images",
            "needs": (
                "labelled photographs with ImageNet-1k class indices (0-999), e.g. a held-out sample of your "
                "own data, scored with top_k_accuracy against the majority-class baseline of that sample"
            ),
        }
    top_k = int(result.get("top_k", DEFAULT_TOP_K))
    ks = sorted({1, min(5, top_k)})
    return {
        **base,
        "metrics": [
            {
                "id": "top_k_accuracy",
                "k": k,
                "value": top_k_accuracy(predictions, list(targets), k=k),
                "estimation": "single sample, no dispersion estimate",
            }
            for k in ks
        ],
        "verdict": "sample-sanity",
        "reason": f"{len(predictions)} labelled image(s) from the tutorial sample; not a benchmark",
        "needs": "a labelled evaluation set from the deployment domain for any generalisable accuracy claim",
    }


@dataclass
class ViTClassificationPipeline:
    """``_runner`` maps a float tensor (N, 3, H, W) to logits (N, NUM_CLASSES); injectable for tests."""

    _runner: Callable[[Any], Any]
    _transform: Callable[[Image.Image], Any]
    device: str = "cpu"
    labels: tuple[str, ...] = ()
    source: str = "injected"
    _feature_runner: Callable[[Any], Any] | None = field(default=None, repr=False)
    classes: list[str] | None = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _head: Any = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTClassificationPipeline:
        import timm
        import torch
        from timm.data import ImageNetInfo, create_transform, resolve_model_data_config

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        arch_name = MODEL_ID.split("/", 1)[1]
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / CONFIG_FILE, encoding="utf-8") as fh:
                config = json.load(fh)
            snapshot_name = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
            if snapshot_name != arch_name:
                raise ValueError(f"snapshot config names {snapshot_name!r}, expected {arch_name!r}")
            overlay = dict(config["pretrained_cfg"])
            overlay["file"] = str(root / WEIGHTS_FILE)  # 'file' takes precedence over hf_hub_id in timm
            model = timm.create_model(
                arch_name, pretrained=True, pretrained_cfg_overlay=overlay, num_classes=NUM_CLASSES
            )
            source = "local-snapshot"
        elif allow_download:
            model = timm.create_model(
                _hub_reference(MODEL_ID, revision=MODEL_REVISION), pretrained=True, num_classes=NUM_CLASSES
            )
            source = "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.eval().to(resolved_device)
        data_config = resolve_model_data_config(model)
        transform = create_transform(**data_config, is_training=False)
        info = ImageNetInfo(subset="imagenet-1k")
        labels = tuple(info.index_to_description(i) for i in range(info.num_classes()))

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))

        def feature_runner(batch: Any) -> Any:
            """Pre-logits: the class token after the final norm, before the ImageNet head (N, FEATURE_DIM)."""
            with torch.inference_mode():
                return model.forward_head(model.forward_features(batch.to(resolved_device)), pre_logits=True)

        return cls(
            runner, transform, resolved_device, labels, source, _feature_runner=feature_runner, _model=model
        )

    def _validate(self, images: Any, top_k: int) -> list[Image.Image]:
        return _check_inputs(images, top_k)

    def predict(
        self, images: Image.Image | Sequence[Image.Image], top_k: int = DEFAULT_TOP_K
    ) -> dict[str, Any]:
        """Classify images; ``score`` is a softmax score over 1000 classes, not a calibrated probability."""
        import torch

        batch_images = self._validate(images, top_k)
        batch = torch.stack([self._transform(image.convert("RGB")) for image in batch_images])
        logits = self._runner(batch)
        if not isinstance(logits, torch.Tensor) or logits.shape != (len(batch_images), NUM_CLASSES):
            raise RuntimeError("runner must return a tensor of shape (batch, NUM_CLASSES)")
        scores = torch.softmax(logits.float(), dim=-1).cpu()
        values, indices = torch.topk(scores, k=top_k, dim=-1)
        predictions = []
        for image_values, image_indices in zip(values.tolist(), indices.tolist(), strict=True):
            image_values = [float(s) for s in image_values]
            ranked = [
                {"label": self.labels[i] if i < len(self.labels) else str(i), "index": i, "score": s}
                for s, i in zip(image_values, image_indices, strict=True)
            ]
            best = ranked[0]
            predictions.append(
                {"predicted_index": best["index"], "predicted_label": best["label"], "top_k": ranked}
            )
        return {
            "predictions": predictions,
            "top_k": top_k,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> Any:
        if self._model is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model

    def _require_head(self) -> tuple[Any, list[str]]:
        if self._head is None or not self.classes:
            raise ValueError("no classification head: call adapt() or load an artifact first")
        return self._head, list(self.classes)

    def _pre_logits(self, images: Sequence[Image.Image]) -> Any:
        """L2-normalised pre-logits (N, FEATURE_DIM) of validated images through the feature runner."""
        import torch

        if self._feature_runner is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        batch = torch.stack([self._transform(image.convert("RGB")) for image in images])
        features = self._feature_runner(batch)
        if not isinstance(features, torch.Tensor) or features.shape != (len(images), FEATURE_DIM):
            raise RuntimeError("feature runner must return a tensor of shape (batch, FEATURE_DIM)")
        return torch.nn.functional.normalize(features.float(), dim=-1).cpu()

    def features(self, records: Sequence[Mapping[str, Any]]) -> Any:
        """Frozen-policy features: one L2-normalised FEATURE_DIM pre-logits vector per validated record."""
        import numpy as np

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        rows = []
        for start in range(0, len(checked), MAX_BATCH):
            rows.append(self._pre_logits([r["image"] for r in checked[start : start + MAX_BATCH]]).numpy())
        return np.concatenate(rows, axis=0).astype(np.float32)

    def knn_baseline(
        self, train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]], *, k: int = 5
    ) -> dict[str, Any]:
        """k-NN over the frozen features of `train`, scored on `test` — the no-training reference point."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import knn_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import class_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        train_checked = validate_dataset(train)["records"]
        test_checked = validate_dataset(test, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        classes = class_names(train_checked)
        return knn_baseline(
            self.features(train_checked),
            [r["label"] for r in train_checked],
            self.features(test_checked),
            [r["label"] for r in test_checked],
            classes,
            k=k,
        )

    def classify(self, images: Image.Image | Sequence[Image.Image]) -> dict[str, Any]:
        """Label validated images with the trained head over the (possibly adapted) backbone's pre-logits —
        the adaptation-contract counterpart of `predict`, which keeps the ImageNet-1k head."""
        import torch

        head, classes = self._require_head()
        batch_images = self._validate(images, DEFAULT_TOP_K)
        head_device = next(head.parameters()).device  # the head lives on the model device while it trains
        with torch.inference_mode():
            logits = head(self._pre_logits(batch_images).to(head_device))
            probabilities = torch.softmax(logits, dim=-1).cpu()
        return {
            "labels": [classes[int(i)] for i in probabilities.argmax(dim=-1)],
            "probabilities": [[float(v) for v in row] for row in probabilities.tolist()],
            "classes": classes,
            "policy": self.adapter["policy"] if self.adapter else "unknown",
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score a validated labelled dataset with the trained head: accuracy, macro-F1, per-class scores."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        _head, classes = self._require_head()
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions: list[str] = []
        nll = 0.0
        index = {c: i for i, c in enumerate(classes)}
        for start in range(0, len(checked), MAX_BATCH):
            chunk = checked[start : start + MAX_BATCH]
            out = self.classify([r["image"] for r in chunk])
            predictions.extend(out["labels"])
            for record, row in zip(chunk, out["probabilities"], strict=True):
                if record["label"] not in index:
                    raise ValueError(f"label {record['label']!r} is not one of the head's classes")
                nll -= math.log(max(row[index[record["label"]]], 1e-12))
        metrics = classification_metrics([r["label"] for r in checked], predictions, classes)
        metrics.update(
            {
                "log_loss": nll / len(checked),
                "policy": self.adapter["policy"] if self.adapter else "unknown",
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_blocks: int) -> list[str]:
        if not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")
        if trainable_blocks == 0:
            return []
        model = self._require_model()
        first = TRANSFORMER_BLOCKS - trainable_blocks
        prefixes = tuple(f"blocks.{k}." for k in range(first, TRANSFORMER_BLOCKS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        probe_steps: int = 300,
        probe_lr: float = 1e-2,
        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
        epochs: int = 4,
        lr: float = 3e-5,
        batch_size: int = 8,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised adaptation of a validated labelled-image dataset in two stages.

        **Frozen policy** (always): a linear head over the frozen, L2-normalised class-token features is
        trained full-batch with AdamW for `probe_steps` steps (`probe_lr`, weight decay 1e-4) — the linear
        probe; epoch 0 of the history records its validation accuracy. **Unfrozen policy** (when
        `trainable_blocks` > 0): the last `trainable_blocks` transformer blocks are unfrozen and trained with
        the head end to end on the images for `epochs` epochs (AdamW at `lr`, weight decay 0.01, gradient
        clipping 1.0, seeded shuffling, no augmentation; the patch embedding, the position embedding, the
        earlier blocks and the final norm stay frozen), scored on validation after every epoch. The epoch with
        the lowest validation log-loss (mean negative log-probability of the gold label) is kept — it may be
        the probe itself; accuracy and macro-F1 are reported beside it at every epoch."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import class_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(probe_steps, int) or not 1 <= probe_steps <= 5_000:
            raise ValueError("probe_steps must be an int in 1..5000")
        if not (0.0 < probe_lr <= 1.0):
            raise ValueError("probe_lr must be in (0, 1]")
        if not isinstance(epochs, int) or not 0 <= epochs <= 20:
            raise ValueError("epochs must be an int in 0..20")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= MAX_BATCH:
            raise ValueError(f"batch_size must be an int in 1..{MAX_BATCH}")
        names = self._trainable_names(trainable_blocks)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        classes = class_names(train_checked)
        import torch

        torch.manual_seed(seed)
        model = self._require_model()
        started = time.perf_counter()
        device = torch.device(self.device)
        index = {c: i for i, c in enumerate(classes)}
        y_train = torch.tensor([index[r["label"]] for r in train_checked], dtype=torch.long)
        # ---- stage A: linear probe on frozen features
        x_train = torch.tensor(self.features(train_checked), dtype=torch.float32)
        head = torch.nn.Linear(FEATURE_DIM, len(classes))
        probe_opt = torch.optim.AdamW(head.parameters(), lr=probe_lr, weight_decay=1e-4)
        probe_losses = []
        for _step in range(probe_steps):
            loss = torch.nn.functional.cross_entropy(head(x_train), y_train)
            probe_opt.zero_grad(set_to_none=True)
            loss.backward()
            probe_opt.step()
            probe_losses.append(float(loss.detach()))
        head.eval()
        self._head, self.classes, self.adapter = head, classes, {"policy": POLICY_FROZEN}

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = ("accuracy", "macro_f1", "log_loss", "n")
            return {k: v for k, v in self.evaluate(val_checked).items() if k in keep}

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {
            "epoch": 0,
            "stage": "linear probe (frozen backbone)",
            "train_loss": probe_losses[-1],
            "val": score_val(),
        }
        history.append(entry)
        if progress:
            progress(entry)
        best_loss = entry["val"]["log_loss"] if entry["val"] else math.inf
        wanted = set(names)
        best_state = {
            "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
            "blocks": {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted},
        }
        best_epoch = 0
        # ---- stage B: bounded unfreeze of the last blocks, end to end with the head
        n_trainable_blocks = 0
        if names and epochs > 0:
            for name, param in model.named_parameters():
                param.requires_grad_(name in wanted)
            block_params = [p for p in model.parameters() if p.requires_grad]
            n_trainable_blocks = sum(p.numel() for p in block_params)
            head.train()
            for p in head.parameters():
                p.requires_grad_(True)
            optimiser = torch.optim.AdamW(
                [{"params": block_params, "lr": lr}, {"params": list(head.parameters()), "lr": lr}],
                weight_decay=0.01,
            )
            generator = torch.Generator().manual_seed(seed)
            tensors = [self._transform(r["image"]) for r in train_checked]
            head.to(device)  # no per-batch GPU<->CPU gradient copies; moved back before the head is used
            y_device = y_train.to(device)
            initial_blocks = {k: v.clone() for k, v in best_state["blocks"].items()}
            try:
                for epoch in range(1, epochs + 1):
                    model.train()
                    head.train()
                    order = torch.randperm(len(train_checked), generator=generator).tolist()
                    losses = []
                    for start in range(0, len(order), batch_size):
                        chosen = order[start : start + batch_size]
                        batch = torch.stack([tensors[i] for i in chosen]).to(device)
                        feats = torch.nn.functional.normalize(
                            model.forward_head(model.forward_features(batch), pre_logits=True).float(), dim=-1
                        )
                        loss = torch.nn.functional.cross_entropy(head(feats), y_device[chosen])
                        optimiser.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(block_params + list(head.parameters()), 1.0)
                        optimiser.step()
                        losses.append(float(loss.detach()))
                    model.eval()
                    head.eval()
                    self.adapter = {"policy": POLICY_UNFROZEN.format(k=trainable_blocks)}
                    entry = {
                        "epoch": epoch,
                        "stage": f"unfrozen last {trainable_blocks} blocks",
                        "train_loss": sum(losses) / max(len(losses), 1),
                        "val": score_val(),
                    }
                    history.append(entry)
                    if progress:
                        progress(entry)
                    current = entry["val"]["log_loss"] if entry["val"] else -math.inf
                    if current < best_loss or not entry["val"]:
                        best_loss = current
                        best_state = {
                            "head": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
                            "blocks": {
                                k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted
                            },
                        }
                        best_epoch = epoch
            except BaseException:
                # Transactional: a failure in training, validation or the progress callback leaves the base
                # exactly as it was, frozen, with no head or adapter attached.
                restore = dict(model.state_dict())
                restore.update(initial_blocks)
                model.load_state_dict(restore, strict=True)
                model.eval()
                for param in model.parameters():
                    param.requires_grad_(False)
                self._head, self.classes, self.adapter = None, [], None
                raise
            head.cpu()
            merged = dict(model.state_dict())
            merged.update(best_state["blocks"])
            model.load_state_dict(merged, strict=True)
            head.load_state_dict(best_state["head"])
            model.eval()
            head.eval()
            for param in model.parameters():
                param.requires_grad_(False)
        for p in head.parameters():
            p.requires_grad_(False)
        policy = POLICY_FROZEN if best_epoch == 0 or not names else POLICY_UNFROZEN.format(k=trainable_blocks)
        self._head, self.classes = head, classes
        self.adapter = {
            "policy": policy,
            "classes": classes,
            "probe_steps": probe_steps,
            "probe_lr": probe_lr,
            "probe_final_loss": probe_losses[-1],
            "trainable_blocks": trainable_blocks if names else 0,
            "trainable_names": names if best_epoch > 0 else [],
            "n_trainable_head": sum(p.numel() for p in head.parameters()),
            "n_trainable_blocks": n_trainable_blocks,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs if names else 0,
            "best_epoch": best_epoch,
            "selection": "lowest validation log-loss (epoch 0 = linear probe)"
            if val_checked
            else "final epoch",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the head (and any trained block tensors) as safetensors with a manifest naming the base."""
        if self.adapter is None or self._head is None:
            raise ValueError("nothing to save: call adapt() first")
        model = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter.get("trainable_names", []))
        tensors = {f"head.{k}": v.detach().cpu().contiguous() for k, v in self._head.state_dict().items()}
        tensors.update(
            {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        )
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHTS_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter.get("history", []),
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(
        self, root: Path, manifest: Mapping[str, Any]
    ) -> tuple[Path, list[str], int]:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, at least two unique classes, a
        canonical policy and an integer `trainable_blocks` in range. Nothing is deserialised here. The
        digest check that follows detects corruption or drift of the weights relative to the adjacent
        manifest; it is not authenticity against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHTS_FILE) != WEIGHTS_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        if not isinstance(adapter, Mapping):
            raise ValueError("artifact manifest has no adapter block")
        classes = list(adapter.get("classes") or [])
        if (
            len(classes) < 2
            or len(set(classes)) != len(classes)
            or not all(isinstance(c, str) for c in classes)
        ):
            raise ValueError("artifact manifest does not name at least two unique classes")
        blocks = adapter.get("trainable_blocks")
        if isinstance(blocks, bool) or not isinstance(blocks, int) or not 0 <= blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(
                f"artifact manifest does not record an integer trainable_blocks in 0..{TRANSFORMER_BLOCKS}"
            )
        policy = adapter.get("policy")
        if policy == POLICY_FROZEN:
            blocks = 0
        elif policy != POLICY_UNFROZEN.format(k=blocks) or blocks == 0:
            raise ValueError(
                f"artifact policy {policy!r} is not a canonical policy for trainable_blocks={blocks}"
            )
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, classes, blocks

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, rebuild
        the head and overlay its block tensors (none under the frozen policy)."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, classes, blocks = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded policy implies: the head, plus the last `blocks` blocks only.
        expected = sorted(["head.bias", "head.weight", *self._trainable_names(blocks)])
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded policy and trainable_blocks")
        model = self._require_model()
        import torch
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        if (
            tuple(tensors.get("head.weight", torch.empty(0)).shape) != (len(classes), FEATURE_DIM)
            or "head.bias" not in tensors
        ):
            raise ValueError("artifact head does not match FEATURE_DIM and the manifest's classes")
        state = model.state_dict()
        block_tensors = {k: v for k, v in tensors.items() if not k.startswith("head.")}
        for key, value in block_tensors.items():
            if key not in state or not key.startswith("blocks."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable transformer-block tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        head = torch.nn.Linear(FEATURE_DIM, len(classes))
        head.load_state_dict({"weight": tensors["head.weight"].float(), "bias": tensors["head.bias"].float()})
        head.eval()
        for p in head.parameters():
            p.requires_grad_(False)
        if block_tensors:
            merged = dict(state)
            merged.update({k: v.to(state[k].dtype) for k, v in block_tensors.items()})
            model.load_state_dict(merged, strict=True)
            model.eval()
        self._head, self.classes = head, classes
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": sorted(block_tensors),
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTClassificationPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/vit_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-image dataset contract for adapting the classifier: the pinned iNaturalist bird sample,
validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and outside the checkpoint's ImageNet-1k label space: 180 CC0-licensed,
research-grade iNaturalist photographs of six common North American birds (30 per species, one per observer
per species), chosen a priori on 2026-09-19 and pinned here by photo id, byte size and SHA-256 of the served
`medium` JPEG. Every file is fetched from the iNaturalist open-data bucket at run time and refused on any
byte-size or SHA-256 mismatch; the repository redistributes none of the photographs. Each record keeps the
observation id and observer login so every image is traceable to its public observation page.

A record is ``{id, image, label}``: a PIL image (or a path to one) and the species key of its gold label.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_IMAGE_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19"
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
CORPUS_BYTES = 19_183_071
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, label, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "song_sparrow-10",
        "song_sparrow",
        637315932,
        349374463,
        "sooji",
        136572,
        "a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033",
        "jpg",
    ),
    (
        "song_sparrow-11",
        "song_sparrow",
        471146686,
        262252507,
        "jeanpaulboerekamps",
        98601,
        "4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b",
        "jpg",
    ),
    (
        "song_sparrow-12",
        "song_sparrow",
        123859010,
        75689904,
        "w_mark_c",
        190819,
        "6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33",
        "jpg",
    ),
    (
        "song_sparrow-13",
        "song_sparrow",
        640811426,
        351179648,
        "erikschiff",
        107695,
        "27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b",
        "jpg",
    ),
    (
        "song_sparrow-14",
        "song_sparrow",
        63537800,
        40010230,
        "nathanael15",
        51441,
        "5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8",
        "jpg",
    ),
    (
        "song_sparrow-15",
        "song_sparrow",
        108520869,
        67204020,
        "dugald",
        52496,
        "e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8",
        "jpg",
    ),
    (
        "song_sparrow-16",
        "song_sparrow",
        435251292,
        244042351,
        "carterdorscht",
        166662,
        "d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a",
        "jpeg",
    ),
    (
        "song_sparrow-17",
        "song_sparrow",
        120710033,
        73898230,
        "tys_rbg",
        126820,
        "515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41",
        "jpg",
    ),
    (
        "song_sparrow-18",
        "song_sparrow",
        393408927,
        222067370,
        "irenemacaulay_",
        101681,
        "22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c",
        "jpg",
    ),
    (
        "song_sparrow-19",
        "song_sparrow",
        349361283,
        198243065,
        "sean579",
        42820,
        "5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074",
        "jpg",
    ),
    (
        "song_sparrow-20",
        "song_sparrow",
        608802621,
        335154467,
        "jamesadney",
        112564,
        "6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513",
        "jpg",
    ),
    (
        "song_sparrow-21",
        "song_sparrow",
        614258628,
        337847585,
        "joy4birds",
        70552,
        "aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67",
        "jpg",
    ),
    (
        "song_sparrow-22",
        "song_sparrow",
        634100352,
        347744524,
        "zorthesosen",
        214044,
        "6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd",
        "jpg",
    ),
    (
        "song_sparrow-23",
        "song_sparrow",
        50065474,
        31954532,
        "truthseqr",
        82521,
        "4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15",
        "jpeg",
    ),
    (
        "song_sparrow-24",
        "song_sparrow",
        8656044,
        6803564,
        "glmory",
        297964,
        "e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5",
        "jpg",
    ),
    (
        "song_sparrow-25",
        "song_sparrow",
        131051149,
        79988590,
        "funvill",
        72572,
        "0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894",
        "jpeg",
    ),
    (
        "song_sparrow-26",
        "song_sparrow",
        12923156,
        9491600,
        "gambolingquail",
        69559,
        "6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d",
        "jpg",
    ),
    (
        "song_sparrow-27",
        "song_sparrow",
        12077500,
        8959545,
        "reuvenm",
        74010,
        "3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593",
        "jpeg",
    ),
    (
        "song_sparrow-28",
        "song_sparrow",
        132730299,
        80955309,
        "steph123456",
        154309,
        "5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270",
        "jpeg",
    ),
    (
        "song_sparrow-29",
        "song_sparrow",
        124840110,
        76316566,
        "terrimewbornagain",
        81046,
        "2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6",
        "jpeg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "chipping_sparrow-10",
        "chipping_sparrow",
        478480946,
        266314208,
        "russnamitz",
        61359,
        "8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b",
        "jpeg",
    ),
    (
        "chipping_sparrow-11",
        "chipping_sparrow",
        220257388,
        129611350,
        "gcart043",
        129377,
        "1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361",
        "jpeg",
    ),
    (
        "chipping_sparrow-12",
        "chipping_sparrow",
        92501489,
        57964052,
        "tniernberger",
        78380,
        "44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5",
        "jpg",
    ),
    (
        "chipping_sparrow-13",
        "chipping_sparrow",
        300376697,
        172991803,
        "matthias55",
        81986,
        "7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b",
        "jpeg",
    ),
    (
        "chipping_sparrow-14",
        "chipping_sparrow",
        58192408,
        36778771,
        "bradenjudson",
        22352,
        "7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75",
        "jpeg",
    ),
    (
        "chipping_sparrow-15",
        "chipping_sparrow",
        80410971,
        50636049,
        "radrat",
        55646,
        "0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667",
        "jpg",
    ),
    (
        "chipping_sparrow-16",
        "chipping_sparrow",
        538480223,
        298914072,
        "hiltonward",
        201271,
        "2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346",
        "jpg",
    ),
    (
        "chipping_sparrow-17",
        "chipping_sparrow",
        370917657,
        209626382,
        "craigmartin",
        101736,
        "feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5",
        "jpg",
    ),
    (
        "chipping_sparrow-18",
        "chipping_sparrow",
        264119989,
        152960401,
        "laurelthrone",
        194107,
        "13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e",
        "jpeg",
    ),
    (
        "chipping_sparrow-19",
        "chipping_sparrow",
        220323755,
        129631264,
        "enspring",
        85687,
        "886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1",
        "jpeg",
    ),
    (
        "chipping_sparrow-20",
        "chipping_sparrow",
        210319996,
        124105091,
        "bunnymom20",
        188328,
        "2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899",
        "jpeg",
    ),
    (
        "chipping_sparrow-21",
        "chipping_sparrow",
        99819947,
        62311494,
        "andy71",
        350329,
        "d649c3c9fd6b0b159a979572b48daba39fc5608104f21c6d88c7d10fa2479f7e",
        "jpg",
    ),
    (
        "chipping_sparrow-22",
        "chipping_sparrow",
        480169680,
        267207764,
        "cvharris",
        144339,
        "d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a",
        "jpeg",
    ),
    (
        "chipping_sparrow-23",
        "chipping_sparrow",
        323624945,
        185313750,
        "mrspteranodon",
        232126,
        "dd60e466086350ce1b9f7a9ba7784fc3963b3f996326ccf52f5adffa5719c39d",
        "jpeg",
    ),
    (
        "chipping_sparrow-24",
        "chipping_sparrow",
        343116928,
        195090930,
        "umamimomma",
        52983,
        "a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2",
        "jpg",
    ),
    (
        "chipping_sparrow-25",
        "chipping_sparrow",
        354617430,
        200940828,
        "wafflemaster135",
        58919,
        "2ba9557d06a7f1bcb1c20908efc82ea6317e5a0bd0c858898f3b5c0a007d20fb",
        "jpeg",
    ),
    (
        "chipping_sparrow-26",
        "chipping_sparrow",
        352953833,
        200088501,
        "aster-asti",
        105786,
        "82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db",
        "jpg",
    ),
    (
        "chipping_sparrow-27",
        "chipping_sparrow",
        465554743,
        259357260,
        "perrydise_koisplash",
        171532,
        "35c6469fad62f198c060e056bd298f87976b39055cf66601a265ab55d5562642",
        "jpeg",
    ),
    (
        "chipping_sparrow-28",
        "chipping_sparrow",
        148831027,
        90084486,
        "ian-wolfe",
        181942,
        "cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49",
        "jpg",
    ),
    (
        "chipping_sparrow-29",
        "chipping_sparrow",
        192945976,
        114256514,
        "sooji",
        200440,
        "04fd7d05d5ced3073d4c7ce8a4f659c6994bb485c5253b26fbc471452b4a0b14",
        "jpeg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        177497964,
        105746665,
        "andywilson",
        29827,
        "65475d4842396f2488167453192d4aa834d2f1b40bcc78b420878c55ccf694d7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "white_throated_sparrow-10",
        "white_throated_sparrow",
        244260317,
        142471526,
        "deejay",
        75694,
        "d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-11",
        "white_throated_sparrow",
        194732675,
        115373159,
        "wildreturn",
        128989,
        "7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831",
        "jpeg",
    ),
    (
        "white_throated_sparrow-12",
        "white_throated_sparrow",
        341990462,
        194541980,
        "ethologist",
        149260,
        "3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb",
        "jpg",
    ),
    (
        "white_throated_sparrow-13",
        "white_throated_sparrow",
        267625208,
        154862375,
        "laurelthrone",
        148616,
        "0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-14",
        "white_throated_sparrow",
        330539586,
        188793537,
        "efalquet",
        119214,
        "c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-15",
        "white_throated_sparrow",
        193091403,
        114346779,
        "ian-wolfe",
        73751,
        "e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479",
        "jpg",
    ),
    (
        "white_throated_sparrow-16",
        "white_throated_sparrow",
        691050773,
        377873006,
        "dinomariobob",
        167744,
        "cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887",
        "jpg",
    ),
    (
        "white_throated_sparrow-17",
        "white_throated_sparrow",
        440986574,
        246671481,
        "suzannehale",
        141857,
        "4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-18",
        "white_throated_sparrow",
        113217820,
        69733707,
        "kemper",
        97802,
        "4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6",
        "jpeg",
    ),
    (
        "white_throated_sparrow-19",
        "white_throated_sparrow",
        575307217,
        318327478,
        "k-simpkins",
        90416,
        "af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b",
        "jpg",
    ),
    (
        "white_throated_sparrow-20",
        "white_throated_sparrow",
        340420076,
        193765555,
        "don54",
        62714,
        "7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556",
        "jpeg",
    ),
    (
        "white_throated_sparrow-21",
        "white_throated_sparrow",
        562344160,
        311510652,
        "rrfc",
        45842,
        "f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0",
        "jpg",
    ),
    (
        "white_throated_sparrow-22",
        "white_throated_sparrow",
        74598266,
        47039878,
        "ianrwhyte",
        141502,
        "b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-23",
        "white_throated_sparrow",
        588001234,
        324825124,
        "portablecity",
        370187,
        "11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607",
        "jpg",
    ),
    (
        "white_throated_sparrow-24",
        "white_throated_sparrow",
        694103481,
        379467387,
        "memoosborne",
        120656,
        "82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a",
        "jpg",
    ),
    (
        "white_throated_sparrow-25",
        "white_throated_sparrow",
        655749108,
        359408087,
        "russnamitz",
        58449,
        "2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1",
        "jpg",
    ),
    (
        "white_throated_sparrow-26",
        "white_throated_sparrow",
        599110899,
        330395670,
        "jd_flores",
        121343,
        "5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae",
        "jpg",
    ),
    (
        "white_throated_sparrow-27",
        "white_throated_sparrow",
        653888173,
        358443775,
        "toknowtheland",
        144197,
        "20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7",
        "jpg",
    ),
    (
        "white_throated_sparrow-28",
        "white_throated_sparrow",
        295037357,
        170132018,
        "sturuss",
        117077,
        "d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3",
        "jpg",
    ),
    (
        "white_throated_sparrow-29",
        "white_throated_sparrow",
        405042885,
        228270930,
        "carterdorscht",
        122157,
        "f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "dark_eyed_junco-10",
        "dark_eyed_junco",
        458710472,
        255914048,
        "joy4birds",
        90973,
        "fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153",
        "jpg",
    ),
    (
        "dark_eyed_junco-11",
        "dark_eyed_junco",
        20784016,
        14046286,
        "gambolingquail",
        93957,
        "a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-12",
        "dark_eyed_junco",
        332842159,
        189933284,
        "jan-konilu",
        110145,
        "d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd",
        "jpeg",
    ),
    (
        "dark_eyed_junco-13",
        "dark_eyed_junco",
        513004508,
        270404136,
        "thevertebratepokedex",
        141252,
        "854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f",
        "jpg",
    ),
    (
        "dark_eyed_junco-14",
        "dark_eyed_junco",
        593499256,
        327583635,
        "orionid",
        108583,
        "7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d",
        "jpg",
    ),
    (
        "dark_eyed_junco-15",
        "dark_eyed_junco",
        63590391,
        40041059,
        "bobbyblackmore",
        82853,
        "87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-16",
        "dark_eyed_junco",
        12580989,
        9282523,
        "artemis224",
        216253,
        "15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093",
        "jpg",
    ),
    (
        "dark_eyed_junco-17",
        "dark_eyed_junco",
        12533281,
        9255403,
        "braincellsgone",
        40857,
        "be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe",
        "jpg",
    ),
    (
        "dark_eyed_junco-18",
        "dark_eyed_junco",
        256948665,
        149140989,
        "igor322",
        86925,
        "9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a",
        "jpeg",
    ),
    (
        "dark_eyed_junco-19",
        "dark_eyed_junco",
        106718005,
        66230973,
        "vicki936",
        41475,
        "e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be",
        "jpeg",
    ),
    (
        "dark_eyed_junco-20",
        "dark_eyed_junco",
        611049594,
        336277421,
        "skylar_schell",
        15425,
        "7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e",
        "jpg",
    ),
    (
        "dark_eyed_junco-21",
        "dark_eyed_junco",
        591147962,
        326403963,
        "toknowtheland",
        106373,
        "07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39",
        "jpg",
    ),
    (
        "dark_eyed_junco-22",
        "dark_eyed_junco",
        469746672,
        261469406,
        "shannon_j",
        74167,
        "ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18",
        "jpg",
    ),
    (
        "dark_eyed_junco-23",
        "dark_eyed_junco",
        459696953,
        256398190,
        "w_mark_c",
        262126,
        "5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c",
        "jpg",
    ),
    (
        "dark_eyed_junco-24",
        "dark_eyed_junco",
        484171775,
        269292238,
        "aschuman",
        59459,
        "89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282",
        "jpg",
    ),
    (
        "dark_eyed_junco-25",
        "dark_eyed_junco",
        357382685,
        202362003,
        "dougbrown",
        56324,
        "70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78",
        "jpeg",
    ),
    (
        "dark_eyed_junco-26",
        "dark_eyed_junco",
        7660371,
        6119391,
        "jeffreyleeisanaturalist",
        108394,
        "0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42",
        "jpg",
    ),
    (
        "dark_eyed_junco-27",
        "dark_eyed_junco",
        437957476,
        245430473,
        "eug302",
        106520,
        "394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-28",
        "dark_eyed_junco",
        339439778,
        193239701,
        "rawcomposition",
        32258,
        "73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a",
        "jpg",
    ),
    (
        "dark_eyed_junco-29",
        "dark_eyed_junco",
        6198359,
        5055484,
        "glmory",
        108168,
        "d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965",
        "jpg",
    ),
    (
        "house_finch-00",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-01",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-02",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-05",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-08",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "house_finch-10",
        "house_finch",
        500744872,
        278868417,
        "pbaff",
        149626,
        "2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0",
        "jpeg",
    ),
    (
        "house_finch-11",
        "house_finch",
        268678834,
        155431721,
        "stevestevens",
        120325,
        "5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79",
        "jpeg",
    ),
    (
        "house_finch-12",
        "house_finch",
        358373136,
        202884575,
        "kcthetc1",
        52329,
        "b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe",
        "jpeg",
    ),
    (
        "house_finch-13",
        "house_finch",
        104227663,
        64793290,
        "verdantpulsar",
        101003,
        "90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5",
        "jpeg",
    ),
    (
        "house_finch-14",
        "house_finch",
        196156834,
        116218899,
        "kgarrett",
        56801,
        "ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0",
        "jpeg",
    ),
    (
        "house_finch-15",
        "house_finch",
        454332148,
        253709711,
        "rlaortiz",
        149985,
        "cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d",
        "jpeg",
    ),
    (
        "house_finch-16",
        "house_finch",
        509969159,
        283798927,
        "damienxw",
        104277,
        "56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805",
        "jpg",
    ),
    (
        "house_finch-17",
        "house_finch",
        665287910,
        295246528,
        "dinomariobob",
        74972,
        "5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49",
        "jpg",
    ),
    (
        "house_finch-18",
        "house_finch",
        168402062,
        100871757,
        "k-simpkins",
        86922,
        "209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d",
        "jpg",
    ),
    (
        "house_finch-19",
        "house_finch",
        247557547,
        144258909,
        "aparrot1",
        79152,
        "115302ef807e6f4d132df584a20ba8644846f05dfe9fee27a7c0e07ec89d87b7",
        "jpg",
    ),
    (
        "house_finch-20",
        "house_finch",
        249874672,
        145517234,
        "vijaybarve",
        115757,
        "15538eeeb228d284f49f33d0bda77626b57fdaabe90d05f4379ffc3bbd855703",
        "jpeg",
    ),
    (
        "house_finch-21",
        "house_finch",
        379972789,
        214941038,
        "dougbrown",
        29723,
        "d7fea293bd3f92aec7be52bdfe5b904793c53cfaeacf9ffd2e762702ee10ba74",
        "jpeg",
    ),
    (
        "house_finch-22",
        "house_finch",
        250140068,
        145607589,
        "matthias55",
        87579,
        "62adc5d86cbdb613779fefa84f6f74f306b6fab5bfb86320f4ef40cd2571ff6a",
        "jpeg",
    ),
    (
        "house_finch-23",
        "house_finch",
        247757548,
        144364266,
        "nana10",
        123890,
        "10b3b6c0b4273a31597050df4218899ddcbe31cdfa60d9a46421ed0bf3d26558",
        "jpeg",
    ),
    (
        "house_finch-24",
        "house_finch",
        253927599,
        147557306,
        "kerykeion",
        91321,
        "6166a9bf59e2075a1129b344a24f3fcafccb610eb984510c5b4c3f9b3abe96af",
        "jpeg",
    ),
    (
        "house_finch-25",
        "house_finch",
        250651171,
        145869416,
        "cathartic_cathartes",
        55197,
        "1929f770ec5ca766c6b88ffdbad5fd7c27df033e12fe46109fd87dc1c835fc59",
        "jpeg",
    ),
    (
        "house_finch-26",
        "house_finch",
        244648136,
        142674679,
        "michelle_lopez",
        51377,
        "9422e42758a68c343d0487d07726819424a24c8271a94aa48c75f4c1e339be77",
        "jpg",
    ),
    (
        "house_finch-27",
        "house_finch",
        373687946,
        211229903,
        "c_dizzy",
        145810,
        "ae7289dc681f8e192886d47018ee70aa9586f08e618e57fa1fe195a5695e1779",
        "jpeg",
    ),
    (
        "house_finch-28",
        "house_finch",
        242992511,
        141794074,
        "chrisleearm",
        102901,
        "b6bd72b41f04d2b6a7855b28e2b169d5aad6663db4a0ac0d364b22ffc2512018",
        "jpg",
    ),
    (
        "house_finch-29",
        "house_finch",
        110518705,
        68278832,
        "kemper",
        103151,
        "62ee726242d1970d9bb8536e31c06635afc92b9352d1394574acb1d9d5eb26ed",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
    (
        "american_goldfinch-10",
        "american_goldfinch",
        143215717,
        86889530,
        "memoosborne",
        129438,
        "ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba",
        "jpeg",
    ),
    (
        "american_goldfinch-11",
        "american_goldfinch",
        403873164,
        227647158,
        "drew_baxter",
        106009,
        "74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49",
        "jpeg",
    ),
    (
        "american_goldfinch-12",
        "american_goldfinch",
        481153019,
        267722510,
        "vicki936",
        255397,
        "3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36",
        "jpg",
    ),
    (
        "american_goldfinch-13",
        "american_goldfinch",
        213311122,
        125763980,
        "hickl",
        24740,
        "0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd",
        "jpeg",
    ),
    (
        "american_goldfinch-14",
        "american_goldfinch",
        417352824,
        234736320,
        "joy4birds",
        81109,
        "357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd",
        "jpeg",
    ),
    (
        "american_goldfinch-15",
        "american_goldfinch",
        72130720,
        45486482,
        "dctphoto",
        178111,
        "17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509",
        "jpeg",
    ),
    (
        "american_goldfinch-16",
        "american_goldfinch",
        45148898,
        28952026,
        "megachile",
        49358,
        "6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252",
        "jpeg",
    ),
    (
        "american_goldfinch-17",
        "american_goldfinch",
        133251099,
        81250513,
        "raffib128",
        128110,
        "863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef",
        "jpeg",
    ),
    (
        "american_goldfinch-18",
        "american_goldfinch",
        155797129,
        93957328,
        "nathanael15",
        74086,
        "7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5",
        "jpg",
    ),
    (
        "american_goldfinch-19",
        "american_goldfinch",
        11400438,
        8535447,
        "akneidel",
        26423,
        "4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b",
        "jpg",
    ),
    (
        "american_goldfinch-20",
        "american_goldfinch",
        145464774,
        88186208,
        "wildreturn",
        68204,
        "565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592",
        "jpg",
    ),
    (
        "american_goldfinch-21",
        "american_goldfinch",
        55990791,
        35505213,
        "conhawn",
        84915,
        "69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb",
        "jpg",
    ),
    (
        "american_goldfinch-22",
        "american_goldfinch",
        637247336,
        289067166,
        "dinomariobob",
        142312,
        "01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd",
        "jpg",
    ),
    (
        "american_goldfinch-23",
        "american_goldfinch",
        460988055,
        257029007,
        "eric112",
        60314,
        "0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9",
        "jpg",
    ),
    (
        "american_goldfinch-24",
        "american_goldfinch",
        59072170,
        37280587,
        "bradenjudson",
        23468,
        "827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42",
        "jpeg",
    ),
    (
        "american_goldfinch-25",
        "american_goldfinch",
        109875643,
        67928198,
        "artemis224",
        217909,
        "99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61",
        "jpg",
    ),
    (
        "american_goldfinch-26",
        "american_goldfinch",
        66515260,
        41915252,
        "reuvenm",
        58833,
        "90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7",
        "jpeg",
    ),
    (
        "american_goldfinch-27",
        "american_goldfinch",
        112715850,
        69464521,
        "umamimomma",
        78717,
        "fae2edcb0901dada461a9ac6b3873d6479e8faf66bfd26a0ad47021f6bcff704",
        "jpg",
    ),
    (
        "american_goldfinch-28",
        "american_goldfinch",
        123422249,
        75440388,
        "rachel_bosley",
        131792,
        "864be3aebb7f1c8c9ed01cdf2e16b690f11354bb37e53b23d44671082afdbcfc",
        "jpg",
    ),
    (
        "american_goldfinch-29",
        "american_goldfinch",
        252964992,
        147051902,
        "carterdorscht",
        149077,
        "a8360d1774df42f034692863781af47fd030265df9bcc1fc938333fabc673c9e",
        "jpeg",
    ),
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 18, "validation": 4, "test": 8}  # per species; 6 species -> 108 / 24 / 48
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MIN_CLASSES = 2
MAX_CLASSES = 100
MAX_LABEL_CHARS = 64
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_RE = re.compile(r"^[A-Za-z0-9_ .:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg or jpeg)."""
    if ext not in ("jpg", "jpeg"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(url, headers={"User-Agent": "dimer-vit-tutorial/1.0"})
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, label}` records with their provenance."""
    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "label": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per species: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for name in out:
        rng.shuffle(out[name])
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/label")
    for key in ("id", "image", "label"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image, label = record["id"], record["image"], record["label"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}"
        )
    if not isinstance(label, str) or not _LABEL_RE.match(label.strip()):
        raise ValueError(
            f"{label_name}: label must be a non-empty string of at most {MAX_LABEL_CHARS} plain characters"
        )
    item = {"id": rid, "image": image.convert("RGB"), "label": label.strip()}
    for key in (
        "source_id",
        "observer",
        "inat_photo_id",
        "inat_observation_url",
        "scientific_name",
        "common_name",
    ):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a labelled-image dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    counts: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        counts[item["label"]] = counts.get(item["label"], 0) + 1
        checked.append(item)
    if not MIN_CLASSES <= len(counts) <= MAX_CLASSES:
        raise ValueError(f"{len(counts)} distinct labels; {MIN_CLASSES}..{MAX_CLASSES} are required")
    sides = [max(r["image"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "classes": sorted(counts),
        "label_counts": dict(sorted(counts.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def class_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted label vocabulary of a dataset (the `classes` a head is trained for)."""
    names = sorted({str(r["label"]) for r in records})
    if len(names) < MIN_CLASSES:
        raise ValueError(f"a dataset needs at least {MIN_CLASSES} distinct labels")
    return names


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image"]), r["label"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    """How many observers contributed photos to more than one split (an observation, not an assertion)."""
    seen: dict[str, set[str]] = {}
    for name, records in splits.items():
        for record in records:
            if record.get("observer"):
                seen.setdefault(str(record["observer"]), set()).add(name)
    return {"observers": len(seen), "in_more_than_one_split": sum(1 for s in seen.values() if len(s) > 1)}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_label.setdefault(record["label"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        n_test = max(1, round(len(pool) * test_fraction))
        n_val = round(len(pool) * val_fraction)
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `labels.csv` (columns `id`, `file`,
    `label`) beside the image files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "labels.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "labels.csv" not in members:
            raise ValueError("BYOD zip must contain labels.csv")
        table = archive.read(members["labels.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding labels.csv and the image files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"labels.csv is missing columns {sorted(missing)}")
    out = []
    for row in rows:
        image = loader(row["file"])
        image.load()
        out.append({"id": row["id"], "image": image.convert("RGB"), "label": row["label"]})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the labels table of a split (id, file, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "file", "label", "observer", "inat_observation_url"]
        )
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record['inat_photo_id']}.jpg" if record.get("inat_photo_id") else record["id"],
                    "label": record["label"],
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e0bd370de679…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ViTClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vit-base-p16-224",
  "modelId": "timm/vit_base_patch16_224.orig_in21k_ft_in1k",
  "revision": "e0bd370de6799e8d1f47a911174ff4c3708e2323",
  "files": [
    {
      "path": "README.md",
      "bytes": 3409,
      "sha256": "c5171b9aefd75755fac887b960b4d1df8cfa1f90568fc03ff5a5d5b74566664d"
    },
    {
      "path": "config.json",
      "bytes": 585,
      "sha256": "91aa54e1244ba735215d4ec117b62c1a00a9f4a744b0e0543b16b901eaf73784"
    },
    {
      "path": "model.safetensors",
      "bytes": 346284714,
      "sha256": "669b949ea91fd19217f200cee259780bde32210c1eb9a5af3859f0dd8346b2ec"
    }
  ],
  "totalBytes": 346288708
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ViTClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the 180 pinned photographs (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is decoded, and `read_corpus` turns each into a `{id, image, label}` record with its species names, observation URL and observer. `build_sample_dataset` draws 18 training, 4 validation and 8 test photographs per species by a seeded stratified shuffle; `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) appears in two splits, `observer_overlap` counts the observers who contributed to more than one split — an observation, since the sample is stratified by species — and the training labels table is written to `outputs/vit_classification_train.csv` in the shape BYOD expects.

Look for: 180 photographs, six classes of 30, splits 108 / 24 / 48, three digests, and four refusal probes — a duplicate id, an oversized image, a single-class dataset and a dataset too small to split — each rejected before `torch` does anything. About a minute on the first run for the downloads.

In [ ]:
import hashlib
import io
import json
import time

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
t0 = time.perf_counter()
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_count = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/inat-birds'))
    raw_count = {'photographs': len(corpus), 'species': len(SPECIES)}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
fetch_seconds = round(time.perf_counter() - t0, 1)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
classes = class_names(train_records)
disjoint = check_split_disjoint(splits)
overlap = observer_overlap(splits)
write_dataset_csv(train_records, 'outputs/vit_classification_train.csv')
print({'data_source': data_source, 'raw': raw_count, 'splits': disjoint, 'classes': classes, 'observer_overlap': overlap, 'fetch_seconds': fetch_seconds, 'corpus_bytes': CORPUS_BYTES})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'label_counts': manifest['label_counts'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {k: example[k] for k in ('id', 'label', 'observer', 'inat_observation_url') if k in example}, 'size': example['image'].size})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'oversized image': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'single class': [{**r, 'label': 'bird'} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Classify through the ImageNet-1k inference contract

Before any adaptation, the inference contract is exercised as it always was, on three test photographs. `validate_inputs` applies exactly the checks `predict` applies — type, batch size 1..`MAX_BATCH`, image side 1..`MAX_IMAGE_SIDE` px, `top_k` 1..`NUM_CLASSES` — and returns an input manifest; a deliberately oversized image is validated too and its rejection recorded as a finding. `predict` returns, per image, `predicted_index` / `predicted_label` and a `top_k` list ordered by descending softmax score under the `argmax` decision rule; the score is a softmax over uncalibrated logits, not a probability of being right. Read the three top-5 lists against the gold species: a goldfinch, a house finch or a junco can be named, because ImageNet-1k has those classes (indices 11, 12, 13); a chipping, song or white-throated sparrow cannot, and the head still answers — the build record saw a chipping sparrow labelled `ruffed grouse` at 0.26 and a song sparrow `brambling` at 0.37. That label-space gap, not a quality defect, is what the rest of the notebook adapts around. The frozen predictions that Section 9 compares against come later, from the probe.

In [ ]:
probe_records = test_records[:3]
print({'ceilings': {'NUM_CLASSES': NUM_CLASSES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}, 'contract': {'INPUT_SIZE': INPUT_SIZE, 'FEATURE_DIM': FEATURE_DIM, 'DECISION_RULE': DECISION_RULE, 'TRANSFORMER_BLOCKS': TRANSFORMER_BLOCKS, 'PARAMETER_COUNT': PARAMETER_COUNT}})
input_manifest = validate_inputs([r['image'] for r in probe_records], top_k=5, names=[r['id'] for r in probe_records])
try:
    validate_inputs(Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/vit_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
imagenet_result = pipe.predict([r['image'] for r in probe_records], top_k=5)
predict_seconds = round(time.perf_counter() - started, 3)
checks = {
    'one_prediction_per_image': len(imagenet_result['predictions']) == 3 and imagenet_result['top_k'] == 5,
    'ranked_descending': all(all(a['score'] >= b['score'] for a, b in zip(p['top_k'], p['top_k'][1:])) for p in imagenet_result['predictions']),
    'argmax_is_rank_one': all(p['predicted_index'] == p['top_k'][0]['index'] for p in imagenet_result['predictions']),
    'scores_in_unit_interval': all(0.0 <= item['score'] <= 1.0 for p in imagenet_result['predictions'] for item in p['top_k']),
}
if not all(checks.values()):
    raise RuntimeError(f'predict output failed a sanity check: {checks}')
print({'probe_ids': [r['id'] for r in probe_records], 'decision_rule': imagenet_result['decision_rule'], 'seconds': predict_seconds, 'device': imagenet_result['device'], 'checks': checks, 'findings': len(input_manifest['findings'])})
for record, prediction in zip(probe_records, imagenet_result['predictions']):
    print({'id': record['id'], 'gold_species': record['label'], 'imagenet_top5': [(item['index'], item['label'].split(',')[0], round(item['score'], 3)) for item in prediction['top_k']]})
print({'note': 'ImageNet-1k names goldfinch (11), house finch (12) and junco (13); it has no class for the three sparrows, and the head answers anyway'})

## 6. The majority floor, the k-NN baseline and the frozen policy

Three numbers frame the adaptation, all on the 48 test photographs. The **majority floor** predicts the most frequent training species for every image — 1 / 6 here, since the sample is balanced. The **cosine 5-NN vote** labels each test image by its five nearest training images in the frozen feature space: what the representation gives with no training at all. The **frozen policy** is `pipe.adapt` with `trainable_blocks=0`: a new six-way linear head over the frozen, L2-normalised 768-d pre-logits (`pipe.features`; the ImageNet head is bypassed, not trained), trained full-batch with AdamW for `PROBE_STEPS` steps — the linear probe — scored on validation as epoch 0 of its history and then on the test split by `pipe.evaluate` (accuracy, macro-F1, per-class recall, confusion). The build record saw the k-NN vote at 75.0 % and the probe at 79.2 %; read the per-class recall to see which species the features confuse — the song sparrow was the weak class. About fifteen seconds on CPU.

In [ ]:
PROBE_STEPS = 300  # @param {type:"integer"}
PROBE_LR = 0.01  # @param {type:"number"}

def brief(m):
    return {'accuracy': round(m['accuracy'], 4), 'macro_f1': round(m['macro_f1'], 4), 'n': m['n']}

floor = majority_baseline([r['label'] for r in train_records], [r['label'] for r in test_records], classes)
print({'majority_floor': brief(floor), 'baseline': floor['baseline']})
t0 = time.perf_counter()
baseline_knn = pipe.knn_baseline(train_records, test_records, k=5)
print({'knn_baseline': brief(baseline_knn), 'baseline': baseline_knn['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
probe_result = pipe.adapt(train_records, val_records, probe_steps=PROBE_STEPS, probe_lr=PROBE_LR, trainable_blocks=0)
frozen_test = pipe.evaluate(test_records)
print({'frozen_policy': probe_result['policy'], 'probe_final_loss': round(probe_result['probe_final_loss'], 4), 'validation': {k: round(v, 4) for k, v in probe_result['history'][0]['val'].items()}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_policy_test': brief(frozen_test), 'log_loss': round(frozen_test['log_loss'], 4), 'verdict': frozen_test['verdict'], 'per_class_recall': {c: round(v['recall'], 2) for c, v in frozen_test['per_class'].items()}})
print({'definitions': frozen_test['definitions']})
assert frozen_test['accuracy'] > floor['accuracy'] and probe_result['policy'].startswith('frozen')

## 7. The unfrozen policy: a bounded unfreeze selected against the probe

`pipe.adapt` with `TRAINABLE_BLOCKS` > 0 first retrains the linear probe on the frozen features (epoch 0 of the history, the frozen policy), then unfreezes the last `TRAINABLE_BLOCKS` transformer blocks — two by default, 14,175,744 of 86,567,656 parameters; the patch embedding, the position embedding, the earlier blocks, the final norm and the ImageNet head stay frozen — and trains them with the new head end to end on the photographs for `EPOCHS` epochs (AdamW at `LEARNING_RATE`, weight decay 0.01, gradient clipping 1.0, seeded shuffling, no augmentation). Every epoch is scored on validation, and the epoch with the **lowest validation log-loss** is kept — epoch 0, the probe, competes on equal terms, so the selected policy can be either. Accuracy and macro-F1 are printed beside the loss at every epoch.

Watch the validation loss: with 24 validation photographs the probe is already near-perfect, so the unfreeze must beat it on confidence, not just on the label. The build record's sweep on this sample: two blocks at 3e-5 for four epochs never beat the probe's validation log-loss (0.162 against 0.163–0.166), so **the probe was selected**; two blocks at 1e-4 was not selected either; four blocks at 1e-4 and two blocks at 3e-4 were selected on validation and lost 6.3 and 12.5 points on the held-out split.

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'stage': entry['stage'], 'train_loss': round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_log_loss'] = round(entry['val']['log_loss'], 4)
        row['val_accuracy'] = round(entry['val']['accuracy'], 4)
        row['val_macro_f1'] = round(entry['val']['macro_f1'], 4)
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, probe_steps=PROBE_STEPS, probe_lr=PROBE_LR, trainable_blocks=TRAINABLE_BLOCKS, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'selected_policy': adapt_result['policy'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'trainable_head': adapt_result['n_trainable_head'], 'trainable_blocks': adapt_result['n_trainable_blocks'], 'total_parameters': adapt_result['n_total'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or policy selection, and no photograph in it appears in the training or validation splits. The selected model is scored exactly as the frozen policy was in Section 6, and the four rows are put side by side: majority floor, k-NN vote, frozen policy, selected policy. Read the policy first: if validation kept the probe, the last two rows are the same model; if it chose the unfreeze, the delta is what the unfreeze bought on 48 photographs — the build record's validation kept the probe at the default settings, and the two unfreezes it did select lost accuracy on the held-out split. The cell asserts the selected model beats the majority floor; it does **not** assert a gain over the probe, because that is the question, not the answer. 48 photographs from one seeded split of one corpus give no dispersion estimate — one image is about two points of accuracy.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {
    metric: {'majority_floor': round(floor[metric], 4), 'knn5': round(baseline_knn[metric], 4), 'frozen_policy': round(frozen_test[metric], 4), 'selected_policy': round(adapted_test[metric], 4)}
    for metric in ('accuracy', 'macro_f1')
}
comparison['log_loss'] = {'frozen_policy': round(frozen_test['log_loss'], 4), 'selected_policy': round(adapted_test['log_loss'], 4)}
comparison['per_class_recall'] = {c: {'knn5': round(baseline_knn['per_class'][c]['recall'], 2), 'frozen': round(frozen_test['per_class'][c]['recall'], 2), 'selected': round(adapted_test['per_class'][c]['recall'], 2)} for c in classes}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('accuracy', 'macro_f1')}
comparison['selected_policy'] = adapt_result['policy']
for metric, row in comparison.items():
    print({metric: row})
print({'confusion_selected': adapted_test['confusion'], 'classes': classes})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'observer_overlap': overlap,
    'classes': classes,
    'baselines': {'majority_floor': floor, 'knn5': baseline_knn},
    'frozen_policy': {'adaptation': {k: v for k, v in probe_result.items() if k not in ('history', 'trainable_names')}, 'history': probe_result['history'], 'test': frozen_test},
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/vit_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['accuracy'] > floor['accuracy']
print({'report': 'outputs/vit_classification_evaluation_report.json'})

## 9. Predict before and after, export the adapter and reload it

Six test photographs are labelled by `pipe.classify` with the selected model and printed beside the frozen policy's predictions (recomputed from the probe's stored head on the frozen features — the head and blocks of the frozen policy were saved before the unfreeze) and the gold species with the top probability; read the probabilities as the head's softmax, not a calibrated confidence. The per-batch `evaluation_report` helper — the inference-stage helper for `predict` — is written for the three probe images and stays `not-measurable`, because the species labels are not ImageNet-1k indices; `pipe.evaluate` is that labelled evaluation.

`pipe.save_artifact` writes the head (`head.weight`, `head.bias`) and, when the unfrozen policy was selected, the trained block tensors — 18.6 KB for the probe alone, about 56.7 MB with two trained blocks — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the classes, the selected policy, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `ViTClassificationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, rebuilds the head from the manifest's classes, refuses any tensor that is not a transformer-block tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The ImageNet head is never in the artifact. The cell asserts identical probabilities on the six images, an identical test accuracy, and identical `predict` output from the reloaded pipeline (VER4).

In [ ]:
import csv
import shutil

show = test_records[:6]
after = pipe.classify([r['image'] for r in show])
frozen_pipe = ViTClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
frozen_pipe.adapt(train_records, val_records, probe_steps=PROBE_STEPS, probe_lr=PROBE_LR, trainable_blocks=0)
before = frozen_pipe.classify([r['image'] for r in show])
rows = []
for i, record in enumerate(show):
    rows.append({'id': record['id'], 'gold': record['label'], 'frozen_label': before['labels'][i], 'frozen_top_probability': round(max(before['probabilities'][i]), 4), 'selected_label': after['labels'][i], 'selected_top_probability': round(max(after['probabilities'][i]), 4), 'observer': record.get('observer', ''), 'inat_observation_url': record.get('inat_observation_url', '')})
    print({k: rows[-1][k] for k in ('id', 'gold', 'frozen_label', 'frozen_top_probability', 'selected_label', 'selected_top_probability')})
single_report = evaluation_report(imagenet_result, sample_kind='three iNaturalist test photographs' if not USE_BYOD else 'three BYOD test images')
print({'batch_report_verdict': single_report['verdict'], 'selected_policy': after['policy'], 'predictions_changed': sum(r['frozen_label'] != r['selected_label'] for r in rows), 'of': len(rows)})
with open('outputs/vit_classification_predictions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/vit_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'vit_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'policy': artifact_manifest['adapter']['policy'], 'classes': artifact_manifest['adapter']['classes'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = ViTClassificationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_after = reloaded.classify([r['image'] for r in show])
reloaded_test = reloaded.evaluate(test_records)
parity = {'probabilities_identical': reloaded_after['probabilities'] == after['probabilities'], 'accuracy_in_memory': round(adapted_test['accuracy'], 6), 'accuracy_reloaded': round(reloaded_test['accuracy'], 6), 'classes_identical': reloaded.classes == pipe.classes, 'predict_identical': reloaded.predict([r['image'] for r in probe_records], top_k=5)['predictions'] == pipe.predict([r['image'] for r in probe_records], top_k=5)['predictions']}
print({'reload_parity': parity, 'reloaded_policy': reloaded.adapter['policy'], 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['probabilities_identical'] and parity['classes_identical'] and parity['predict_identical'] and abs(adapted_test['accuracy'] - reloaded_test['accuracy']) < 1e-9

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHTS_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'photographs': len(SAMPLE_RECORDS), 'bytes': CORPUS_BYTES, 'license': CORPUS_LICENSE, 'species': SPECIES},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_ids': [r['id'] for r in probe_records], 'imagenet_top1': [p['predicted_label'] for p in imagenet_result['predictions']], 'seconds': predict_seconds},
    'comparison': comparison,
    'predictions_before_after': rows,
    'batch_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'policy': artifact_manifest['adapter']['policy']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/vit_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen ViT-B/16 pre-logits — trained to separate 1000 ImageNet classes, three of which are these birds — already separate all six species well enough that a cosine 5-NN vote reaches 75 % and a linear probe 79 % on 48 held-out photographs against a majority floor of one in six, and a bounded unfreeze of the last two blocks on 108 training photographs never beat the probe's validation log-loss, so validation kept the probe. That is the claim and the finding: the adaptation contract gives the checkpoint a label space it did not have, runs both policies end to end on a real labelled corpus, chooses between them on validation rather than by assumption, and reports the answer against a floor and a no-training baseline rather than in isolation.

The test split is 48 photographs from one seeded split of one small corpus with no dispersion estimate — one image is about two points of accuracy, so a two-point delta is noise. Observers contribute to more than one split (the notebook counts them), so some of what the head learns may be a photographer's style rather than a bird. Accuracy and macro-F1 say whether the gold species is predicted, not whether the features are good for any other task; the head's softmax is not a calibrated confidence. When the unfrozen policy is selected it changes the last blocks, which every input shares, so `predict` — which keeps the untouched ImageNet head — returns different ImageNet-1k outputs after it; the artifact records which policy won.

Three things to carry to real data. **Floors first:** the majority floor and the k-NN vote on *your* images are the numbers to read before any trained head's — if the probe barely beats k-NN, the features already carry the task. **Leakage:** split by photographer, session or device when your images come from one (the contract de-duplicates by pixels, not by source). **Policy:** unfreezing is a hypothesis to test on validation, not a default; with a few hundred images the probe is usually the honest choice — here it was — and the two unfreezes that validation did select in the build sweep lost six and twelve points on the held-out split.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled image corpus, validate the demonstrated dataset contract without leakage, execute the ImageNet-1k inference contract, a linear probe and a bounded unfreeze with validation-based policy selection, evaluate by accuracy and macro-F1 against a floor and a k-NN baseline on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, ImageNet-1k accuracy, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_BLOCKS = 4` and `LEARNING_RATE = 1e-4` and read what validation selects and what the held-out split says about it (the build record: selected at epoch 4, 72.9 % on test); set `LEARNING_RATE = 3e-4` for a hotter unfreeze (selected at epoch 2, 66.7 %); set `EPOCHS = 8` and watch whether validation log-loss ever drops below the probe's; or bring your own labelled images through BYOD and read the k-NN baseline before either policy.

## References

- Repository README: https://github.com/kurtvalcorza/vit-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/vit-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/vit-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/timm/vit_base_patch16_224.orig_in21k_ft_in1k
- Upstream code: https://github.com/huggingface/pytorch-image-models
- An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (Dosovitskiy et al., 2020): https://arxiv.org/abs/2010.11929
- iNaturalist open data (CC0 photographs credited to their observers in the carried records): https://www.inaturalist.org/pages/developers
- timm documentation: https://huggingface.co/docs/timm
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)